# 06-workflow.ipynb

In [55]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
llm = init_chat_model('gpt-4.1-mini')

## Prompt Chaining

- 매우 잘 정리된 업무 순서가 있을 경우 사용
- 이전 노드에서 처리한 내용을 `state`에 담아 다음 노드로 전송

In [56]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 그림 보는 용
from IPython.display import Image, display

# Graph State
class State(TypedDict):
    topic: str
    joke: str
    improved_joke: str
    final_joke: str

# Nodes
def generated_joke(state: State):
    msg = llm.invoke(f'주제 {state['topic']}에 관련된 짧은 농답 생성')
    return {'joke': msg.content}

def improve_joke(state: State):
    msg = llm.invoke(f'말장난을 추가해서 아래 농담을 더 재밌게 만들어보자. \n {state['joke']}')
    return {'improved_joke': msg.content}    

def polish_joke(state: State):
    msg = llm.invoke(f'아래 농담을 재미있게 뒤틀어 보자. \n {state['improved_joke']}')
    return {'final_joke': msg.content}

# Router
def check_punchline(state: State):
    # ? 나 !가 없으면 농담을 improve 하고, 있으면 그대로 진행
    if '?' in state['joke'] or '!' in state['joke']:
        return 'Pass'
    else:
        return 'Fail'
    
# 조립
workflow = StateGraph(State)

workflow.add_node(generated_joke)
workflow.add_node(improve_joke)
workflow.add_node(polish_joke)

workflow.add_edge(START, 'generated_joke')

workflow.add_conditional_edges(
    'generated_joke',
    check_punchline,
    {
        'Pass': END,
        'Fail': 'improve_joke'
    }
)

workflow.add_edge('improve_joke', 'polish_joke')
workflow.add_edge('polish_joke', END)

graph = workflow.compile()

graph

graph.invoke({'topic': '랭체인'})


{'topic': '랭체인',
 'joke': '랭체인?  \n언어의 사슬이라길래,  \n내 말도 연결해주나 했더니,  \nAI가 내 생각을 줄줄이 엮어주네!'}

## Parallelization (병렬화)
- 여러 node(llm)이 동시에 작업을 진행
- 뭐가 먼저 끝날지 알 수 없음
- 하위 task를 동시에 진행시켜서 속도를 올린다.
- 같은 task를 여러번 동시에 돌려서 신뢰성 확보

In [57]:
class State(TypedDict):
    topic : str
    joke: str
    story: str
    poem: str
    output: str

# Nodes
def generate_joke(state: State):
    msg = llm.invoke(f'Write a joke about {state['topic']}')
    return {'joke': msg.content}

def generate_story(state: State):
    msg = llm.invoke(f'Write a story about {state['topic']}')
    return {'story': msg.content}

def generate_poem(state: State):
    msg = llm.invoke(f'Write a poem about {state['topic']}')
    return {'poem': msg.content}

def aggregate(state: State):
    output = f"""농담, 이야기, 시
    Joke: {state['joke']}
    Story: {state['story']}
    Poem: {state['poem']}
    """    
    return {'output': output}


workflow = StateGraph(State)
workflow.add_node(generate_joke)
workflow.add_node(generate_story)
workflow.add_node(generate_poem)
workflow.add_node(aggregate)

workflow.add_edge(START, 'generate_joke')
workflow.add_edge(START, 'generate_story')
workflow.add_edge(START, 'generate_poem')

workflow.add_edge('generate_joke', 'aggregate')
workflow.add_edge('generate_story', 'aggregate')
workflow.add_edge('generate_poem', 'aggregate')

workflow.add_edge('aggregate', END)

graph = workflow.compile()
graph.invoke({'topic': '피자'})

{'topic': '피자',
 'joke': '왜 피자가 운동하러 갔을까요?  \n치즈가 늘어나야 하니까요! 🧀🍕😄',
 'story': '옛날 어느 작은 마을에 피자를 사랑하는 소년 태훈이 살고 있었어요. 태훈은 매일 학교가 끝나면 동네 피자 가게 ‘황금 도우’에 들러 맛있는 피자를 먹는 것이 행복이었죠.\n\n어느 날, 가게 주인 할아버지가 태훈에게 특별한 비밀을 알려줬어요. “이 피자는 그냥 피자가 아니란다. 황금 도우에는 사랑과 정성이 담겨 있어, 그것을 먹는 사람에게 기쁨과 용기를 주지.” 태훈은 할아버지의 말을 듣고 감동했어요.\n\n그날부터 태훈은 자신도 사람들에게 행복을 전하는 피자를 만들고 싶다는 꿈을 갖게 되었어요. 집에서 엄마와 함께 여러 가지 토핑을 올리고 반죽을 직접 돌돌 말아서 ‘태훈 피자’를 만들기 시작했죠. 친구들도 그의 피자를 먹고는 모두 웃음을 지었어요.\n\n마침내 마을 축제 날, 태훈은 ‘황금 도우’ 앞에서 직접 만든 피자를 나눠주었어요. 사람들은 태훈의 피자를 한 입 먹자마자 기운이 솟고, 마음이 따뜻해지는 걸 느꼈어요. 그날 이후로 태훈의 피자는 마을 사람들의 사랑을 듬뿍 받으며 모두의 행복을 이어주는 특별한 음식이 되었답니다.\n\n그리고 태훈은 어른이 되어도 변치 않는 마음으로, 사랑과 정성이 담긴 피자를 세상에 퍼뜨리고 있답니다. 끝.',
 'poem': 'Golden crust with edges bright,  \nCheese that melts in warm delight,  \nTomato sauce, a tangy trace,  \n피자 smiles on every face.  \n\nToppings piled in colors bold,  \nPepperoni, peppers rolled,  \nMushrooms, olives, herbs that please,  \nHarmony in every piece.  \n\nOven’s kiss, a fragrant glow,  \nSlices shared, c

In [58]:
from typing import Literal
from langchain.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

class State(TypedDict):
    input: str
    decision: str
    output: str

# Router 에서 사용할 LLM이 쓸 structured output
# {'step': 'poem' | 'story' | 'joke'}
class Route(BaseModel):             
    # step 이라는 필드는 세 개중 하나지만, 필수(...)고 설명은 description임.
    step: Literal['poem', 'story', 'joke'] = Field(..., description='next step in routing process') #Field(...)은 필수로 있어야 된다는 뜻

router_llm = llm.with_structured_output(Route)

# state['decision']을 결정하는 Node
def decision_node(state: State):    
    result = router_llm.invoke(
        [
            SystemMessage(content='사용자 요청에 따라 story, joke, poem을 선택'),
            HumanMessage(content=state['input'])
        ]
    )
    return {'decision': result.step}

# Nodes
def generate_joke(state: State):
    msg = llm.invoke([
        SystemMessage(content='아주 기발하되 고급스러운 유머를 만들어야 함.'),
        HumanMessage(content=state['input'])
    ])
    return {'output': msg.content}

def generate_story(state: State):
    msg = llm.invoke(state['input'])
    return {'output': msg.content}

def generate_poem(state: State):
    msg = llm.invoke(state['input'])
    return {'output': msg.content}


# Router
def route_decision(state: State):
    return state['decision']

workflow = StateGraph(State)
workflow.add_node(decision_node)
workflow.add_node(generate_joke)
workflow.add_node(generate_story)
workflow.add_node(generate_poem)

workflow.add_edge(START, 'decision_node')

workflow.add_conditional_edges(
    'decision_node',
    route_decision,
    {
        'joke': 'generate_joke',
        'story': 'generate_story',
        'poem': 'generate_poem'
    }
)

workflow.add_edge('generate_joke', END)
workflow.add_edge('generate_story', END)
workflow.add_edge('generate_poem', END)
graph = workflow.compile()

graph
graph.invoke({'input': '심심해'})

/Users/jinny/sesac/05-langgraph/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Route(step='joke'), input_type=Route])
  return self.__pydantic_serializer__.to_python(


{'input': '심심해',
 'decision': 'joke',
 'output': '그럼 이렇게 한 번 웃어보시죠:\n\n“아인슈타인이 커피를 너무 좋아해서 ‘상대성 이론’ 대신 ‘상대성 카페인’을 만들었다는데, 알고 보니 그건 단지 커피 타임마다 시간이 더 빨리 간다는 뜻이었다고 하네요.”\n\n심심함을 잠시라도 달래드렸길 바랍니다! 더 기발한 유머도 필요하시면 말씀해 주세요.'}

## Orchestrator - Worker
- task를 sub-task로 쪼갬
- 해당 sub-task를 담당하는 worker에게 분배
- 모든 sub-task들의 output을 최종 결과로 정리

In [59]:
from typing import Annotated, List
from pydantic import BaseModel, Field

# Structured Output 용 Model 생성 (계획 수립용)
class Section(BaseModel):
    name: str = Field(
        ...,
        description='보고서내 해당 섹션명'
    )
    description: str = Field(
        ...,
        description='해당 섹션에서 다룰 주요 주제 및 개념에 대한 개요'
    )

class Sections(BaseModel):
    sections: List[Section] = Field(
        ...,
        description='리포트의 섹션들'
    )

planner_llm = llm.with_structured_output(Sections)

# planner_llm.invoke('LLM 스케일링 법칙에 대한 레포트 작성해줘.')


In [60]:
import operator
from langgraph.types import Send # 유동적으로 worker 노드를 실행할 수 있게 해줌.

# Graph State
class State(TypedDict):
    topic: str
    sections: list[Section]
    completed_sections: Annotated[list, operator.add]
    final_report: str

# Worker State
class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]

# Nodes
def orchestrator(state: State):
    report_sections = planner_llm.invoke(
        [
            SystemMessage(content='generate a plan for report.'),
            HumanMessage(content=f'Here is the imort topic: {state['topic']}')
        ]
    )

    return {'sections': report_sections.sections}

def worker(state: WorkerState):
    section = llm.invoke(
        [
            SystemMessage(content='제공된 name과 description에 따라 보고서 섹션을 작성해라. 각 섹션 앞에는 서두를 넣지말고, 마크다운 형식으로 작성.'),
            HumanMessage(content=f'섹션 name: {state['section'].name} \n 섹션 description: {state['section'].description}')
        ]
    )
    return {'completed_sections': [section.content]}


def synthesizer(state: State):
    # 단순히 worker들의 결과 합치기
    completed_sections = state['completed_sections']
    final_report = '\n\n -------- \n\n'.join(completed_sections)
    return {
        'final_report': final_report
    }


# Router
def assign_workers(state: State):
    plan = []
    for s in state['sections']:
        plan.append(Send('worker', {'section': s}))
    
    return plan

# Graph
workflow = StateGraph(State)

workflow.add_node(orchestrator)
workflow.add_node(worker)
workflow.add_node(synthesizer)

workflow.add_edge(START, 'orchestrator')

workflow.add_conditional_edges(
    'orchestrator',
    assign_workers,
    ['worker']
)
workflow.add_edge('worker', 'synthesizer')
workflow.add_edge('synthesizer', END)

graph = workflow.compile()

graph
# display(Image(graph.get_graph().draw_mermaid_png))

result = graph.invoke({'topic': 'RAG에 대한 논문 작성'})

/Users/jinny/sesac/05-langgraph/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Sections(sections=[Sectio...연구 방향 제안')]), input_type=Sections])
  return self.__pydantic_serializer__.to_python(


In [61]:
from IPython.display import Markdown
Markdown(result['final_report'])

# 서론

RAG( Retrieval-Augmented Generation)는 외부 지식을 활용하여 텍스트 생성의 정확성과 다양성을 향상시키는 최신 인공지능 기술입니다. 전통적인 자연어 처리 모델은 주어진 데이터에만 의존하지만, RAG는 방대한 외부 데이터베이스에서 관련 정보를 검색한 후 이를 바탕으로 문장을 생성함으로써 더 신뢰할 수 있는 결과를 도출합니다. 이러한 특성은 정보의 신속한 업데이트와 복잡한 질문에 대한 응답 능력을 크게 향상시킵니다.

최근 정보량이 급증하고 사용자의 요구가 다양해짐에 따라, 단순한 데이터 기반 생성 모델은 한계에 봉착하게 되었습니다. 따라서 RAG와 같은 하이브리드 모델의 연구가 필수적이며, 이는 자연어 처리 분야의 혁신적인 발전을 도모할 수 있는 중요한 전환점이 될 것입니다. 본 연구에서는 RAG의 기본 개념을 살펴보고, 그 필요성과 활용 가능성에 대해 다각도로 분석하고자 합니다.

 -------- 

# 관련 연구

최근 RAG(Retrieval-Augmented Generation)와 관련된 연구가 활발히 진행되고 있다. RAG는 대규모 언어 모델이 외부 지식 베이스를 참조하여 보다 정확하고 풍부한 응답을 생성할 수 있도록 하는 기술로, 정보 검색과 자연어 생성의 융합 분야에 속한다.

대표적인 논문으로는 Lewis et al.(2020)의 "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"가 있으며, 이 연구에서는 사전 학습된 생성 모델과 검색 모델을 결합해 지식 집약적 태스크에서 성능 향상을 보여주었다. 또한, RAG 모델은 위키피디아와 같은 방대한 문서 집합에서 필요한 정보를 실시간으로 검색해 답변의 정확도를 높이는 특징이 있다.

기술 동향 측면에서는 RAG 기반 모델들이 어휘 기반 검색을 넘어 신경망 기반의 밀집 표현 검색(dense retrieval)을 도입하면서 발전하고 있다. 이와 함께, 효율적인 검색 및 생성 전략, 멀티모달 데이터 확장, 지식 업데이트 메커니즘 등이 연구 주제로 부상하고 있다.

한편, 상업적 응용 분야에서도 RAG는 고객 지원 챗봇, 문서 요약, 맞춤형 정보 제공 등에 활용되며 점차 실용성을 인정받고 있다. 기존 연구들은 RAG가 자연어 이해와 생성의 한계를 극복하는 핵심 기술로 자리매김하는 데 기여하고 있음을 시사한다.

 -------- 

# RAG 개념 및 원리

RAG는 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합한 기술입니다. 이 접근법은 대규모 언어 모델이 사전에 학습한 지식뿐만 아니라 외부 데이터베이스나 문서에서 실시간으로 관련 정보를 검색하여 더 정확하고 신뢰할 수 있는 답변을 생성할 수 있도록 합니다.

작동 원리는 크게 두 단계로 나눌 수 있습니다. 첫째, 사용자의 질문이나 입력에 대해 관련된 자료를 검색하는 Retrieval 단계입니다. 이 단계에서는 키워드 매칭, 벡터 유사도 탐색 등 다양한 검색 기법이 활용됩니다. 둘째, 검색된 자료를 바탕으로 자연어 생성 모델이 답변을 작성하는 Generation 단계입니다. 이 때, 모델은 검색된 정보를 통합하여 맥락에 맞고 풍부한 답변을 생성합니다.

이러한 방식은 대규모 언어 모델의 한계인 최신 정보 부족 문제를 보완하며, 신뢰성과 정확성을 높여 다양한 응용 분야에서 효율적으로 활용됩니다.

 -------- 

# RAG의 적용 사례

RAG(Retrieval-Augmented Generation)는 다양한 분야에서 기존 자연어 처리 모델의 한계를 극복하고 성능을 향상시키기 위해 활용되고 있다. 주요 실제 적용 사례는 다음과 같다.

## 1. 고객 지원 자동화
기업들은 RAG를 활용해 방대한 제품 매뉴얼과 고객 응대 데이터를 기반으로 고객 질문에 대한 적절한 답변을 자동 생성한다. 특히 복잡한 문제 해결에 필요한 세부 정보를 문서에서 검색하고, 이를 바탕으로 자연스러운 응답을 제공함으로써 고객 만족도를 높였다.

## 2. 의료 문헌 검색 및 요약
의료 분야에서 RAG는 신규 연구 논문이나 임상 가이드라인 등 방대한 의료 문헌에서 관련 정보를 신속하게 찾아내어 요약하는 데 사용된다. 이를 통해 의사와 연구자는 최신 정보를 효율적으로 습득할 수 있으며, 정확한 진단 및 치료 결정을 지원받는다.

## 3. 법률 문서 분석
법률 분야에서는 판례, 법률 조항, 계약서 등 다양한 문헌을 대상으로 RAG를 적용하여 특정 사건 관련 법률 정보를 추출하고 해석하는 데 활용된다. 이를 통해 변호사들은 복잡한 법률 문제에 대한 근거 자료를 신속하게 확보할 수 있다.

## 4. 교육 콘텐츠 생성
교육 분야에서는 학생들이 질문하는 내용에 대해 교과서와 참고 자료를 기반으로 정확하고 상세한 답변을 생성하는 데 RAG가 사용된다. 이는 맞춤형 학습과효율적인 자기주도 학습 환경 조성에 기여한다.

전반적으로 RAG는 복잡한 정보를 필요로 하는 다양한 산업과 분야에서 정보 검색과 텍스트 생성 능력을 결합하여 혁신적인 솔루션을 제공하는 데 핵심적인 역할을 수행하고 있다.

 -------- 

## 성능 평가 및 비교

RAG(Retrieval-Augmented Generation)의 성능 평가는 일반적으로 다음과 같은 방법들을 통해 수행된다. 먼저, 정보 검색의 정확도와 문서 검색률을 측정하여 RAG가 적절한 관련 문서를 얼마나 잘 찾아내는지를 평가한다. 이후, 생성된 응답의 정확성, 유창성, 일관성 등을 평가하기 위해 자동 평가 지표(예: BLEU, ROUGE, METEOR)와 인간 평가를 병행한다. 또한, 응답의 관련성과 사실성(factuality)을 검증하는 별도의 평가도 이루어진다.

다른 기술과의 비교 연구에서는 RAG와 전통적인 순수 생성 모델, 단순 검색-응답 병합 모델, 그리고 최신의 대형 언어 모델과의 성능을 비교한다. 일반적인 비교 기준으로는 응답의 질, 처리 속도, 자원 소모, 확장성 등이 포함된다. 연구 결과, RAG는 검색된 외부 지식 기반으로부터 정보를 효과적으로 통합하여 정확도와 사실성을 개선하는 데에 유리한 것으로 나타났다. 반면, 단순 생성 모델은 창의성에서 강점을 가지나 사실 검증 측면에서 한계가 있다. 최신 대형 언어 모델은 방대한 사전 학습으로 강력한 성능을 보이나, 최신 정보 반영과 특정 도메인 지식 활용에서는 RAG의 검색 기반 접근법이 장점을 가지는 경우가 많다. 

종합적으로, RAG는 정보 검색과 텍스트 생성의 장점을 결합하여 다양한 자연어 처리 태스크에서 우수한 성능을 발휘하며, 다른 기술들과 비교할 때 현실 세계 지식 활용 측면에서 특히 유리한 방법임이 입증되었다.

 -------- 

# 결론 및 향후 연구 방향

이번 연구에서는 주요 변수들 간의 상관관계를 분석하고, 특정 요인이 결과에 미치는 영향을 체계적으로 조사하였다. 실험과 데이터 분석을 통해 도출된 결과는 기존 이론을 보완하는 동시에 실제 적용 가능성을 확인하는 데 중요한 기초 자료를 제공하였다. 특히, 본 연구에서 제안한 모델은 높은 예측 정확도와 안정성을 보여주어 관련 분야의 연구 발전에 기여할 것으로 기대된다.

향후 연구에서는 다음과 같은 방향을 제안한다. 첫째, 적용 대상과 범위를 확대하여 다양한 환경과 조건에서 모델의 일반화 능력을 검증할 필요가 있다. 둘째, 본 연구에서 다루지 않은 변수들을 포함하여 복합적인 상호작용 효과를 분석함으로써 연구의 깊이를 더할 수 있을 것이다. 셋째, 최신 알고리즘과 기술을 도입하여 분석 방법론을 고도화하고, 실시간 데이터 처리와 적용 가능성을 탐색하는 데 중점을 둬야 한다. 이를 통해 본 연구의 성과를 기반으로 보다 폭넓고 실질적인 연구 성과를 창출할 수 있을 것으로 기대된다.

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel, Field
from typing import Annotated, List, Literal
from tavily import TavilyClient
from langchain.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage, SystemMessage
import operator
from langgraph.types import Send # 유동적으로 worker 노드를 실행할 수 있게 해줌.
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent




llm = init_chat_model('gpt-4.1-mini')

TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

class Section(BaseModel):
    name: str = Field(
        ...,
        description='보고서내 해당 섹션명'
    )
    description: str = Field (
        ...,
        description='해당 섹션에서 다룰 주요 주제 및 개념에 대한 개요'
    )

class Sections(BaseModel):
    sections: List[Section] = Field(
        ...,
        description='리포트의 섹션들'
    )


planner_llm = llm.with_structured_output(Sections)





@tool
def internet_search(
        query:str,
        max_results: int = 5, # = 뒤에는 기본값
        topic: Literal['general', 'news', 'finance'] = 'general',
        include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic
    )



class State(TypedDict):
    topic: str
    sections: list[Section]
    completed_sections: Annotated[list, operator.add]
    final_report: str

class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]


def orchestrator(state: State):
    report_sections = planner_llm.invoke(
        [
            SystemMessage(content='generate a plan for report.'),
            HumanMessage(content=f'Here is the imort topic: {state['topic']}')
        ]
    )

    return {'sections': report_sections.sections}






AGENT_SYSTEM_PROMPT = """
사용자가 요청한 주제에 대해 리포트를 작성하기 위한 source 검색용으로 internet_search tool 사용.
"""

agent = create_agent(
    llm,
    tools=[internet_search],
    system_prompt=AGENT_SYSTEM_PROMPT

)

def worker(state: WorkerState): # WorkerState를 인자로 받음




    message = f"섹션명: {state['section'].name} \n설명: {state['section'].description}\n위 주제에 대해 조사해서 리포트 내용을 작성해줘."
    result = agent.invoke({
        'messages': [
            HumanMessage(content=message)
        ]
    })

    # 결과 messages안에 Human, AI, Tool 등등 다 빼고 마지막 AI 메시지의 내용만 사용
    return {"completed_sections": [result['messages'][-1].content]}




    


def synthesizer(state: State):
    # 단순히 worker들의 결과 합치기
    completed_sections = state['completed_sections']
    final_report = '\n\n -------- \n\n'.join(completed_sections)
    return {
        'final_report': final_report
    }


# Router
def assign_workers(state: State):
    plan = []
    for s in state['sections']:
        plan.append(Send('worker', {'section': s}))
    
    return plan
    
    

In [22]:

# Graph
workflow = StateGraph(State)

workflow.add_node(orchestrator)
workflow.add_node(worker)
workflow.add_node(synthesizer)

workflow.add_edge(START, 'orchestrator')

workflow.add_conditional_edges(
    'orchestrator',
    assign_workers,
    ['worker']
)
workflow.add_edge('worker', 'synthesizer')
workflow.add_edge('synthesizer', END)

graph = workflow.compile()

graph
# display(Image(graph.get_graph().draw_mermaid_png))

result = graph.invoke({'topic': '2025년 10월에 업데이트 된 Langgraph에 대한 논문 작성'})

/Users/jinny/sesac/05-langgraph/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Sections(sections=[Sectio...적인 결론 도출')]), input_type=Sections])
  return self.__pydantic_serializer__.to_python(


In [23]:
from IPython.display import Markdown
Markdown(result['final_report'])

서론

LangGraph는 LangChain의 일부로, LLM(대형 언어 모델) 애플리케이션의 흐름을 그래프로 정의하고 실행할 수 있는 라이브러리입니다. 기존의 선형 체인 방식을 넘어서 상태 머신과 그래프 개념을 도입함으로써, 더 복잡하고 유연한 워크플로우를 구성할 수 있다는 점에서 주목받고 있습니다. 특히 LangGraph는 엔드 유저가 직접 로컬 환경에서 에이전트를 구성하고 제어할 수 있도록 설계되어, 프롬프트나 RAG 기반 접근법과 달리 사용자 주도의 프로세스 흐름 제어가 가능합니다.

2025년 10월 예정된 LangGraph의 대규모 업데이트는 이 라이브러리의 활용성 및 기능성을 대폭 향상시키는 중요한 전환점이 될 것으로 기대됩니다. 이번 업데이트에서는 기존의 create_react_agent 기능이 확장되어, 후처리 및 전처리 훅을 보다 편리하게 활용할 수 있게 되었고, Runtime(Context) 활용 능력이 강화되어 정적 및 동적 컨텍스트 관리가 분리, 체계적으로 가능해졌습니다. 이러한 변화는 LangGraph를 기반으로 하는 LLM 에이전트 개발의 자유도와 실용성을 크게 높여, 복잡한 시스템 설계와 실행 관리에서 사용자의 편의성을 증대시키는 중요한 의미를 가집니다.

따라서 LangGraph와 2025년 10월 업데이트는 LLM 애플리케이션 설계 및 구현 생태계에 깊은 영향을 미치는 혁신적 이정표로 자리매김할 것이며, 이를 심도 있게 이해하는 것은 최신 AI 기술 동향을 선도하는 데 필수적이라 할 수 있습니다.

 -------- 

섹션명: 관련 연구  
설명: 기존 Langgraph 연구 및 기술 동향 정리

1. LangGraph 개요  
LangGraph는 복잡한 AI 에이전트 작업을 안정적으로 수행할 수 있는 에이전트 런타임 및 저수준 오케스트레이션 프레임워크로, 복잡한 워크플로우를 구성하고 멀티 에이전트 시스템을 구축하는 데에 최적화되어 있다. LangGraph는 프로그램 로직의 복잡성 해소를 위해 흐름 제어를 추상화하고, 순환 및 비선형 그래프 형태의 복잡한 작업 흐름을 처리할 수 있다는 점에서 기존 프레임워크와 차별화된다.

2. 주요 연구 및 적용 사례  
- LangGraph는 RAG(Retrieval-Augmented Generation)를 포함한 다양한 AI 에이전트 조합을 손쉽게 구현할 수 있는 drag-and-drop 사용자 인터페이스를 제공하여 코드 작성 없이 복잡한 에이전트 워크플로우 설계가 가능하다.  
- 다중 역할의 AI 에이전트들이 협력하여 콘텐츠 생성, 프로젝트 계획 등 다양한 업무를 자동화하는 시스템 구축에 활용된다.  
- 인공지능 분야 연구 및 학술기관에서는 LangGraph를 활용해 고도화된 멀티 에이전트 연구 파이프라인, 교육 도구, 재현 가능한 실험 프레임워크 구축에 활용한다.

3. 개발자 커뮤니티 및 기술 트렌드  
- LangGraph는 복잡하고 비선형적인 에이전트 작업 처리를 가능하게 하는 유연성을 바탕으로 다양한 산업의 개발자들에게 인기를 끌고 있다. 특히, 기존의 단순한 DAG(Directed Acyclic Graph) 기반 툴들과 달리 순환 구조를 포함한 복잡한 그래프 흐름을 지원함으로써 고난이도 문제 해결에 적합하다는 평가를 받고 있다.  
- 개발자들 사이에서는 LangGraph가 복잡한 흐름 제어 로직을 추상화해주어 개발 생산성을 크게 높인다는 점과, 사용자가 원하는 맞춤형 에이전트 설계가 가능하다는 점에서 긍정적인 반응이 많다.

4. 연구 에이전트 및 오케스트레이션 발전  
- LangGraph는 연구 에이전트 구현에 필수적인 도구 호출 및 상태 관리, 중간 결과 저장 등의 기능을 통해 다양한 도구 및 API와 긴밀한 연동을 지원한다.  
- 오픈소스 및 상용 LLM(대규모 언어 모델)과 결합하여 신뢰성 높은 AI 에이전트 구현의 기반 기술로 자리잡고 있으며, 생산 환경에서의 확장성과 즉시 디버깅, 빠른 반복 개발을 지원한다.

정리하면, LangGraph는 다중 AI 에이전트의 복잡한 상호작용을 효과적으로 관리하고, 고급 연구 및 산업용 AI 시스템 개발을 가속화하는 에이전트 오케스트레이션의 핵심 프레임워크로서 최신 AI 기술 트렌드를 반영하고 있다. 향후 인공지능을 활용한 자동화 및 지능형 시스템 구축 분야에서 더욱 중요한 역할을 수행할 전망이다.

 -------- 

리포트: 2025년 10월 업데이트 내용

1. 개요
2025년 10월은 주요 소프트웨어 및 운영체제에서 중대한 변화가 발생한 시기입니다. 특히, Microsoft의 Windows 10 운영체제와 관련된 공식 지원 및 업데이트 정책이 종료되면서 많은 사용자가 Windows 11로의 전환을 고려해야 하는 상황이 되었습니다.

2. Windows 10 지원 종료 및 영향
- 2025년 10월 14일부로 Microsoft는 Windows 10 운영체제에 대한 공식적인 무료 소프트웨어 업데이트, 보안 수정, 기술 지원을 종료합니다.
- 이는 Windows 10에 대한 보안 업데이트, 버그 수정 및 기술 지원이 더 이상 제공되지 않는다는 의미로, 보안 취약점 노출 위험이 커집니다.
- Microsoft는 Windows 11로의 업그레이드를 권장하며, 최소 시스템 요구 사항을 충족하는 PC에 대해 업그레이드 알림을 제공하고 있습니다.
- 확장 보안 업데이트(ESU) 프로그램을 통해 최대 1년간 추가 보안 업데이트를 제공하나, 이는 임시 방편에 불과합니다.
- Microsoft 365 앱에 대한 Windows 10 지원도 2025년 10월 14일에 종료되며, 관련 보안 업데이트는 최대 3년간 Windows 11 환경에서만 지원됩니다.

3. 보안 및 기능 개선
- 2025년 10월에 갤럭시 S25 시리즈 등 일부 모바일 기기에 보안 패치가 배포되어 보안 관련 안정화 코드가 적용되었습니다.
- 보안 정책이 강화되어, 업데이트 이후에는 보안이 낮은 이전 소프트웨어 버전으로의 다운그레이드가 제한됩니다.

4. 기술적 변화 및 시사점
- Windows 10 지원 종료는 많은 기업과 개인 사용자에게 운영체제 업그레이드 또는 새로운 기기 구매를 촉진하는 계기가 됩니다.
- 장기적으로 보안 위협을 최소화하고, 최신 기능과 안정적인 시스템 환경을 제공하기 위한 필수적인 조치입니다.
- 사용자는 업데이트 종료 후에도 Windows 10 시스템을 계속 사용할 수 있으나 보안 위험에 노출될 수 있으므로, 신속한 Windows 11 이전이 권고됩니다.

5. 결론
2025년 10월은 Windows 10 운영체제에 대한 공식적인 무료 지원 종료가 이루어진 중요한 시기로, 보안과 기능 면에서 큰 변화가 있었습니다. 사용자와 기업은 이에 대비하여 운영체제 업그레이드 및 보안 강화 조치를 취해야 하며, 최신 소프트웨어 환경으로의 전환이 필수적입니다.

참고 자료
- Microsoft 공식 지원 페이지 (Windows 10 지원 종료 안내)
- 갤럭시 S25 시리즈 2025년 10월 보안 패치 공지
- 관련 IT 뉴스 및 커뮤니티 게시글

 -------- 

섹션명: 적용 사례  
설명: 업데이트된 Langgraph의 실제 적용 사례 및 효과 분석

---

1. 개요  
업데이트된 Langgraph는 복잡한 작업 흐름을 그래프 구조로 정의하고, 에이전트 간 상호작용과 상태 관리를 효율적으로 관리할 수 있는 기능을 강화하였다. 이를 통해 다양한 분야에서 멀티 에이전트 시스템을 구축하고 복잡한 문제를 해결하는 데 효과적으로 활용되고 있다.

2. 실제 적용 사례

- 멀티 에이전트 워크플로우 관리  
  Langgraph는 노드와 엣지를 기반으로 한 유연한 그래프 구조를 사용하여 여러 에이전트 간 데이터 흐름과 의사소통을 관리한다. 이를 통해 복잡한 비즈니스 로직을 단계별로 명확히 나누고, 상태 추적 및 비동기 병렬 처리도 원활하게 지원한다.  
  예를 들어, 금융, 제조, 헬스케어 등 다양한 산업에서 다단계 의사결정 프로세스에 적용하여 효율성을 높이고 있다.

- 휴먼 인 더 루프(HITL) 에이전트 구축  
  Langgraph와 Elasticsearch를 결합해 사용자가 의사결정 과정에 참여하여 문맥의 공백을 메우고, 자동화된 도구 호출 전에 검토하는 HITL 애플리케이션이 개발되고 있다.  
  이 방식은 내결함성이 중요하고 복잡한 데이터 해석이 필요한 법률, 고객지원, 의료 분야 등에 적합하며, 사용자 참여를 통해 정확도와 신뢰성을 크게 향상시킨다.

- LLM 기반 애플리케이션 개발  
  Langgraph는 상태 관리 기능을 통해 각 노드의 상태를 추적하면서 LLM(대형 언어 모델) 기반 애플리케이션을 효율적으로 구동한다. 예컨대, 반복 학습 및 평가 과정을 자동화하고, 최적의 모델 파라미터 탐색에 활용되어 머신러닝 워크플로우 자동화에 적용된다.  

3. 효과 분석

- 작업 흐름 복잡성 감소 및 유지보수 용이  
  그래프 기반 워크플로우 구조 덕분에 복잡한 작업 흐름을 시각적으로 분석하고, 중복 작업을 줄여 시스템 유지보수가 쉬워졌다.

- 확장성 및 병렬 처리 향상  
  멀티 에이전트 구성을 지원해 대규모 애플리케이션에 적용 가능하며, 비동기 처리 기능 덕분에 동시 작업 처리 속도가 향상되었다.

- 내결함성 및 신뢰성 증가  
  HITL 방식으로 사용자가 의사결정에 직접 참여 가능하여 자동화의 한계를 보완하고, 전반적인 시스템 신뢰성을 높이고 있다.

---

이와 같이 업데이트된 Langgraph는 복잡한 데이터 흐름과 에이전트 간 상호작용을 효율적으로 관리하며, 다양한 산업 분야에서 실제 적용되어 긍정적인 효과를 보이고 있다. 특히 유연한 그래프 구조와 강력한 상태 관리 기능, 휴먼 인 더 루프 통합은 차별화된 경쟁력으로 평가받고 있다.

 -------- 

섹션명: 성능 평가
설명: 업데이트 전후 성능 비교 및 평가 결과 제시

리포트 내용:

성능 평가는 시스템이나 모델의 변화가 실제 성능에 어떠한 영향을 미쳤는지를 객관적으로 파악하는 중요한 과정입니다. 특히 업데이트 전후의 성능 비교는 최신 버전의 개선 효과를 검증하고, 문제점 또는 성능 저하 여부를 확인하는 데 필수적입니다.

1. 평가 방법 및 지표 선정
업데이트 전후 성능 비교를 위해서는 일관성 있고 신뢰할 수 있는 평가 지표가 필요합니다. 대표적인 성능 평가 지표로는 처리 시간, 정확도, 응답 속도, 자원 사용률 등이 있으며, 평가 대상에 따라 적절한 지표를 선정해야 합니다. 예를 들어 AI 모델의 경우 정확도(Accuracy), 정밀도(Precision), 재현율(Recall) 등 다양한 지표가 사용됩니다.

2. 비교 수행 절차
업데이트 이전 버전과 이후 버전에 동일한 평가 데이터셋을 투입하여 결과를 산출하고 비교 분석합니다. 이 과정에서 평가 환경, 테스트 조건을 동일하게 유지하여 외부 변수 영향을 최소화하는 것이 중요합니다.

3. 결과 해석 및 시각화
성능 개선 정도를 명확히 이해하기 위해 표, 그래프 등을 사용하여 수치 변화를 시각화합니다. 예를 들어 처리 속도가 얼마나 빨라졌는지, 오류율이 얼마나 감소했는지 등 주요 지표 변화를 명확하게 제시해야 합니다.

4. 평가 사례 및 고려 사항
- 평가 일관성 확보: 업데이트 후 평가 결과가 이전과 달라 일관성이 무너질 경우 객관적 결과 도출이 어렵습니다.
- 성능 저하 원인 분석: 업데이트 후 일부 성능 지표가 악화될 수 있으므로 원인 분석과 추가 개선 계획 수립 필요.
- 지속적 모니터링: 한 차례 평가에 그치지 않고 지속적으로 성능을 모니터링하여 안정성을 확보하는 것이 권장됩니다.

결론적으로, 업데이트 전후 성능 평가를 체계적이고 객관적으로 수행함으로써 개선 효과를 명확히 확인하고, 향후 시스템 및 모델 개선 방향을 수립할 수 있습니다. 이 과정은 개발팀뿐 아니라 경영진, 이해관계자들에게 중요한 의사결정 근거를 제공합니다.

 -------- 

향후 연구 방향 - LangGraph 발전을 위한 연구 가능성과 제안

1. 내구적 실행(Durable Execution)의 고도화  
LangGraph는 내구적 실행 기능을 지원하여 에이전트가 실패 후에도 상태를 보존하고 장시간 작동할 수 있도록 설계되었다. 향후 연구에서는 이 내구적 실행 기능을 더욱 정교화하고, 분산 환경에서의 내구성 보장, 자가 복구 및 자가 최적화 기능을 추가하는 방향으로 발전시킬 수 있다.

2. 인간-인-루프(Human-in-the-loop) 시스템의 심화  
현재 LangGraph는 인간이 에이전트의 상태를 실시간으로 점검하고 수정할 수 있는 기능을 제공한다. 연구를 통해 인간과 AI 에이전트 간의 상호작용 인터페이스를 개선하고, 더 자연스럽고 효율적인 피드백 루프를 구축하는 데 집중할 수 있다. 예를 들어, 사용자 맞춤형 제어와 투명성 강화 연구가 필요하다.

3. 복잡한 작업 및 장기 플래닝 지원  
LangGraph는 그래프 기반 구조를 활용하여 다단계 및 다중 에이전트 워크플로우를 관리한다. 향후 연구는 복잡한 작업을 더 잘 처리할 수 있도록 장기 플래닝, 동적 의사결정, 메타학습, 그리고 에이전트 간 협업 메커니즘을 통합하는 방안을 모색할 수 있다.

4. 실시간 스트리밍 및 인터럽트 처리 최적화  
실시간 입력을 반영하고, 필요 시 에이전트 실행을 중단하거나 변경할 수 있는 스트리밍과 인터럽트 기능이 LangGraph에는 포함되어 있다. 이 기능을 강화해 비정형 데이터 처리 그리고 외부 이벤트에 대한 즉각적인 반응성을 높이는 연구가 요구된다.

5. 멀티모달 에이전트 통합 및 도메인 특화  
언어 모델 중심의 LangGraph를 기반으로 이미지, 음성, 비디오 등 멀티모달 데이터를 처리하는 에이전트를 통합하고, 의료, 법률, 금융 등 특정 도메인에 특화된 워크플로우와 에이전트 모델을 설계하는 연구가 필요하다.

6. 에이전트 행동의 투명성 및 해석 가능성 향상  
LangGraph는 그래프 형태로 에이전트 상태와 행위를 시각화하지만, 복잡한 의사결정 과정을 더 명확히 설명하고 검증 가능한 체계로 만드는 인터프리터블 AI 연구가 병행되어야 한다.

7. 에코시스템 확장 및 API 고도화  
현재 API와 툴 세트를 바탕으로 다양한 AI 솔루션 개발이 가능하지만, 확장성, 상호운용성, 그리고 사용자 친화적 개발 환경 제공을 위해 에코시스템을 지속 확장하고 API 기능을 고도화하는 연구가 중요하다.

요약하자면, LangGraph의 발전을 위해서는 내구성, 인간-인-루프 상호작용, 복잡도 처리 능력, 실시간 반응성, 멀티모달 통합, 투명성, 그리고 개발자 경험 향상을 목표로 한 연구들이 주도적으로 이루어져야 하며, 이를 통해 더 강력하고 신뢰할 수 있는 AI 에이전트 개발 플랫폼으로 자리매김할 수 있을 것이다.

 -------- 

[결론 섹션: 논문의 주요 내용 요약 및 종합적인 결론 도출]

논문의 결론은 연구 전체를 마무리하는 핵심 부분으로서, 연구의 주요 내용을 간결하고 명확하게 요약하는 동시에, 종합적인 결론을 도출하는 역할을 합니다. 효과적인 결론 작성법을 정리하면 다음과 같습니다.

1. 연구 결과 요약
   - 논문에서 다룬 연구 목적과 질문을 다시 상기시키고, 핵심적인 연구 결과를 간략히 정리합니다.
   - 연구 과정에서 밝혀진 중요한 발견이나 성과를 짧고 명확하게 서술합니다.

2. 연구의 의의 및 적용 가능성 제시
   - 연구 결과가 현장이나 실무, 학문 분야에 어떤 영향을 미칠 수 있는지를 설명합니다.
   - 결과가 가지는 이론적, 실용적 의미를 포함해 실제 적용 아이디어 등을 제안합니다.

3. 연구의 한계와 향후 연구 방향 제안
   - 본 연구에서 제한되었던 부분, 미처 다루지 못한 점들을 솔직하게 언급합니다.
   - 이러한 한계가 이후 연구에서 어떻게 보완될 수 있는지, 후속 연구가 나아갈 방향을 제시하여 연구의 확장 가능성을 보여줍니다.

4. 효과적인 문장 구성
   - 불필요한 반복을 자제하고, 간결하면서도 논문의 주제에 부합하도록 자연스럽게 작성합니다.
   - 연구 내용을 과장하거나 모호하게 표현하지 않고, 객관적이고 명확한 언어를 사용합니다.

이처럼 결론은 단순한 요약을 넘어서 연구의 가치를 명확히 하고, 독자가 연구의 의의를 충분히 이해할 수 있도록 정리하는 섹션입니다. 또한 향후 연구에 대한 방향성을 제시함으로써 학술적 대화의 연속성을 유지하는 중요한 역할을 합니다.